# ⚡ GIADA — IE-P4 · Aggiornamento ionico discreto congiunto
Formule vettorizzate, LUT e12reti (capacità×supervisione×seed). dt0,1ms,17gate espliciti; SK, calcio, readout, sinapsi e solver invariati.
Freeze su development prima dei fresh atomici e24path imposti. Task35 è diagnostica storica, non nuova conferma biologica. GPU e Internet; nessun dataset esterno.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working/giada_ie_p4');WORK.mkdir(parents=True,exist_ok=True)
REPO=WORK/'giada';BRANCH='codex/surrogate-validity-audit'
if not REPO.exists():subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
REF=os.environ.get('GIADA_IE_P4_REVISION',BRANCH)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REF],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
print({'revision':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),'protocol':'IE-P4','training':True})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_ie_p4_discrete_ionic')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}'
LOG=WORK/'process.log'
with subprocess.Popen([sys.executable,'-u',str(REPO/'scripts/run_ie_p4_discrete_ionic.py'),str(OUTPUT)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as worker,LOG.open('w') as log:
    for line in worker.stdout:
        log.write(line);log.flush()
        if line.startswith('[GIADA IE-P4]'):print(line.rstrip(),flush=True)
    exit_code=worker.wait()
OUTPUT.mkdir(parents=True,exist_ok=True);shutil.copy2(LOG,OUTPUT/'process.log')
path=next((OUTPUT/n for n in ('final_report.json','failure_report.json') if (OUTPUT/n).is_file()),None)
REPORT=json.loads(path.read_text()) if path else {'valid':False,'error':'worker terminated without report','exit':exit_code}
if path is None:(OUTPUT/'failure_report.json').write_text(json.dumps(REPORT,indent=2))
print({k:REPORT.get(k) for k in ('valid','error','device','candidate_promoted')});print({'rows':len(REPORT.get('rows',[])),'graph_failures':len(REPORT.get('graph_failures',[]))})
if exit_code:print('\n'.join(LOG.read_text(errors='replace').splitlines()[-18:]))
ARCHIVE=shutil.make_archive('/kaggle/working/giada_ie_p4_discrete_ionic','zip',root_dir=OUTPUT)


In [ ]:
import base64
from IPython.display import Javascript, display
payload=base64.b64encode(Path(ARCHIVE).read_bytes()).decode()
display(Javascript('const s=atob('+json.dumps(payload)+');const b=Uint8Array.from(s,c=>c.charCodeAt(0));const u=URL.createObjectURL(new Blob([b],{type:"application/zip"}));const a=document.createElement("a");a.href=u;a.download="giada_ie_p4_discrete_ionic.zip";document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),10000);'))
